# 🚕 NYC Taxi Lakehouse

## Silver Layer - Data Cleansing & Standardization

The purpose of the Silver layer is to transform raw Bronze data into a clean, consistent, and analytics-ready dataset.

Unlike the Bronze layer, this stage applies data quality checks, derives useful attributes, and prepares the data for downstream business reporting.

### Objectives

- Read data from the Bronze layer
- Profile the dataset
- Identify data quality issues
- Apply business rules
- Create derived columns
- Store the cleaned dataset as a Delta table

In [0]:
from pyspark.sql import functions as F

## Read Bronze Table

We'll begin by loading the raw Delta table created in the Bronze layer.

All transformations in this notebook will use the Bronze table as the source.

In [0]:
bronze_df = spark.table("taxi.bronze.yellow_taxi")

In [0]:
display(bronze_df.limit(10))

## Data Profiling

In [0]:
#We'll first review the size of the dataset and inspect the schema.

print(f"Total Rows    : {bronze_df.count()}")
print(f"Total Columns : {len(bronze_df.columns)}")

In [0]:
bronze_df.printSchema()

## Missing Values

Understanding missing values helps determine whether records should be removed, imputed, or left unchanged.

For the Silver layer, we'll identify which columns contain null values before defining any cleaning rules.

In [0]:
display(
    bronze_df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in bronze_df.columns
    ])
)

## Duplicate Records

Duplicate records can distort reporting and KPI calculations.

Let's determine whether the dataset contains any exact duplicate rows.

In [0]:
total_rows = bronze_df.count()

distinct_rows = bronze_df.distinct().count()

print(f"Total Rows     : {total_rows:,}")
print(f"Distinct Rows  : {distinct_rows:,}")
print(f"Duplicates     : {total_rows - distinct_rows:,}")

## Pickup Date Range

Reviewing the pickup date range confirms the reporting period covered by the dataset.

In [0]:
display(
    bronze_df.select(
        F.min("tpep_pickup_datetime").alias("Minimum Pickup"),
        F.max("tpep_pickup_datetime").alias("Maximum Pickup")
    )
)

## Fare Amount Analysis

Negative fares generally indicate invalid records or adjustment transactions.

We'll inspect whether any exist before defining our cleaning rules.

In [0]:
bronze_df.filter(
    F.col("fare_amount") < 0
).count()

In [0]:
display(
    bronze_df.filter(
        F.col("fare_amount") < 0
    )
)

## Trip Distance Analysis

Trip distance should normally be greater than zero.

We'll investigate records with negative or zero distances.

In [0]:
print(
    "Negative Distance:",
    bronze_df.filter(
        F.col("trip_distance") < 0
    ).count()
)

print(
    "Zero Distance:",
    bronze_df.filter(
        F.col("trip_distance") == 0
    ).count()
)

## Timestamp Validation

A trip cannot end before it begins.

Let's verify whether any records violate this rule.

In [0]:
invalid_time = bronze_df.filter(
    F.col("tpep_dropoff_datetime") <
    F.col("tpep_pickup_datetime")
)

print(
    f"Invalid Trips : {invalid_time.count():,}"
)

## Passenger Count Distribution

Passenger count is an important business attribute.

Reviewing its distribution helps identify unusual or unexpected values.

In [0]:
display(
    bronze_df.groupBy("passenger_count")
             .count()
             .orderBy("passenger_count")
)

## Payment Type Distribution

Understanding payment methods provides insight into customer behavior and helps validate categorical values.

In [0]:
display(
    bronze_df.groupBy("payment_type")
             .count()
             .orderBy("payment_type")
)

#Ratecode Distribution

In [0]:
display(
    bronze_df.groupBy("RatecodeID")
             .count()
             .orderBy("RatecodeID")
)

#Vendor Distribution

In [0]:
display(
    bronze_df.groupBy("VendorID")
             .count()
             .orderBy("VendorID")
)

# Apply Data Quality Rules

The Bronze layer stores the raw data exactly as it was received from the source system.

Based on the data profiling performed earlier, we'll now apply a small set of quality rules to remove records that would negatively impact downstream analytics.

### Rules Applied

- Remove records with negative fare amounts.
- Remove records where the dropoff time occurs before the pickup time.

No additional filtering is applied at this stage because values such as zero-distance trips, unknown passenger counts, and unknown payment types are considered valid within the NYC TLC dataset.

In [0]:
silver_df = (
    bronze_df
        .filter(F.col("fare_amount") >= 0)
        .filter(
            F.col("tpep_dropoff_datetime") >=
            F.col("tpep_pickup_datetime")
        )
)

## Validate Cleaning Results

Let's verify that the quality rules were successfully applied before moving on to feature engineering.

In [0]:
print("Rows Before :", bronze_df.count())
print("Rows After  :", silver_df.count())

In [0]:
print(
    "Negative Fare Remaining:",
    silver_df.filter(
        F.col("fare_amount") < 0
    ).count()
)

print(
    "Invalid Timestamp Remaining:",
    silver_df.filter(
        F.col("tpep_dropoff_datetime") <
        F.col("tpep_pickup_datetime")
    ).count()
)

# Create Date & Time Features

Many business reports analyze trends over time.

Instead of repeatedly extracting date components in every dashboard query, we'll derive them once in the Silver layer.

These attributes will support daily, monthly, hourly, and weekday analyses in the Gold layer.

In [0]:
silver_df = (
    silver_df
        .withColumn(
            "pickup_date",
            F.to_date("tpep_pickup_datetime")
        )
        .withColumn(
            "pickup_year",
            F.year("tpep_pickup_datetime")
        )
        .withColumn(
            "pickup_month",
            F.month("tpep_pickup_datetime")
        )
        .withColumn(
            "pickup_day",
            F.dayofmonth("tpep_pickup_datetime")
        )
        .withColumn(
            "pickup_hour",
            F.hour("tpep_pickup_datetime")
        )
        .withColumn(
            "pickup_day_name",
            F.date_format(
                "tpep_pickup_datetime",
                "EEEE"
            )
        )
)

In [0]:
display(silver_df)

# Create Business Features

The Silver layer is responsible for enriching the dataset with reusable business attributes.

Rather than calculating these metrics repeatedly in dashboard queries, we derive them once and store them in the Silver layer.

The features created in this section will support reporting, KPI calculations, and business analysis in the Gold layer.

In [0]:
    ## Trip Duration

silver_df = silver_df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime") -
        F.unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

In [0]:
## Tip Percentage

silver_df = silver_df.withColumn(
    "tip_percentage",
    F.when(
        F.col("fare_amount") > 0,
        F.round(
            (F.col("tip_amount") / F.col("fare_amount")) * 100,
            2
        )
    ).otherwise(None)
)

In [0]:
## Average Trip Speed

silver_df = silver_df.withColumn(
    "trip_speed_mph",
    F.when(
        F.col("trip_duration_minutes") > 0,
        F.round(
            F.col("trip_distance") /
            (F.col("trip_duration_minutes") / 60),
            2
        )
    ).otherwise(None)
)

In [0]:
## Weekend Indicator

silver_df = silver_df.withColumn(
    "is_weekend",
    F.when(
        F.dayofweek("pickup_date").isin(1, 7),
        True
    ).otherwise(False)
)

In [0]:
## Time of Day

silver_df = (
    silver_df
    .withColumn(
        "day_period",
        F.when(F.col("pickup_hour").between(5,11), "Morning")
         .when(F.col("pickup_hour").between(12,16), "Afternoon")
         .when(F.col("pickup_hour").between(17,20), "Evening")
         .otherwise("Night")
    )
)

In [0]:
display(
    silver_df.select(
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "trip_duration_minutes",
        "trip_speed_mph",
        "tip_percentage",
        "pickup_hour",
        "is_weekend",
        "day_period"
    )
)

In [0]:
## Trip Category

silver_df = (
    silver_df.withColumn(
        "trip_category",
        F.when(F.col("trip_distance") < 2, "Short")
         .when(F.col("trip_distance") < 5, "Medium")
         .when(F.col("trip_distance") < 10, "Long")
         .otherwise("Very Long")
    )
)

In [0]:
display(silver_df.select("trip_category").limit(10))

In [0]:
# Storing into Silver table

SILVER_TABLE = "taxi.silver.yellow_taxi"

(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_TABLE)
)

In [0]:
%sql
select count(*) from taxi.silver.yellow_taxi

# Silver Layer Summary

The Silver layer has been successfully created.

### Transformations Applied

- Removed records with negative fare amounts
- Removed records with invalid timestamps
- Added reusable date and time attributes
- Calculated trip duration
- Calculated tip percentage
- Calculated average trip speed
- Added weekend indicator
- Categorized trips by time of day
- Classified trips by distance

The dataset is now standardized, enriched, and ready for business aggregation in the Gold layer.